# Monitoring a model service — the hands-on half

The practical companion to **`monitoring_slides.html`**. By the end of this notebook there is a
real Prometheus scraping a real instrumented API, a real Grafana dashboard, and a real alert rule
that fires — all in containers, all torn down at the end.

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 3–4 | check Docker, sandbox, the app to instrument |
| 1 · Instrument | 5–9 | counters, histograms, gauges — and what each is for |
| 2 · Scrape | 10 | Prometheus in a container, actually collecting |
| 3 · Query | 11 | PromQL: rate, percentiles, error ratio — against real data |
| 4 · Dashboard | 12–13 | Grafana, provisioned from files, screenshotted |
| 5 · Alerts & drift | 14–18 | an alert rule that fires, and what ML adds to all this |

**Why this session is different:** sessions 3 and 4 got a model into production. This one is about
the six months afterwards, when nothing crashes but the numbers quietly stop being right.

Everything runs in a throwaway `mon_demo/` folder; the last cell stops the stack and removes it.

<!-- ar -->
<div dir="rtl" lang="ar">

**مراقبة خدمة نموذج: الجزء العملي**

هذا الدفتر هو الجانب العملي لعرض `monitoring_slides.html`. في نهايته سيكون عندك Prometheus حقيقي يجمع مقاييس API حقيقية، ولوحة Grafana حقيقية، وقاعدة تنبيه حقيقية تنطلق، وكلها في حاويات، وكلها تُزال في النهاية.

| الجزء | شرائح العرض | ماذا تفعل |
|---|---|---|
| ٠ · الإعداد | 3–4 | فحص Docker، ومجلد التجربة، والتطبيق الذي سنراقبه |
| ١ · إضافة المقاييس | 5–9 | العدّادات، والمدرّجات، والمقاييس اللحظية، وفائدة كل منها |
| ٢ · الجمع | 10 | Prometheus داخل حاوية، يجمع فعلاً |
| ٣ · الاستعلام | 11 | PromQL: المعدل، والنسب المئوية، ونسبة الأخطاء، على بيانات حقيقية |
| ٤ · اللوحة | 12–13 | Grafana، مُعدّة من ملفات، مع صورة لها |
| ٥ · التنبيهات والانحراف | 14–18 | قاعدة تنبيه تنطلق، وما يضيفه تعلم الآلة لكل هذا |

**لماذا هذه الجلسة مختلفة:** الجلستان ٣ و ٤ أوصلتا النموذج إلى الإنتاج. هذه الجلسة عن الأشهر الستة التالية، حين لا ينهار شيء لكن الأرقام تتوقف عن كونها صحيحة بهدوء.

كل شيء يعمل داخل مجلد مؤقت اسمه `mon_demo/`، وآخر خلية توقف الحاويات وتحذفه.

</div>

## Step 0.1 · Docker check

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫١ · فحص Docker**

</div>

In [1]:
!docker --version && docker compose version
print()
!docker info --format 'server {{.ServerVersion}}, {{.NCPU}} cpus' 2>/dev/null || echo "daemon not reachable"

Docker version 29.6.1, build 8900f1d
Docker Compose version v5.3.0

server {.ServerVersion}, {.NCPU} cpus


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · فحص Docker**

بنطبع نسخة Docker ونسخة `docker compose` عشان نتأكد إنهم مثبتين. بعدين بنسأل الخدمة الخلفية لـ Docker عن نسختها وعدد الأنوية، ولو مش شغالة بنطبع إنها مش متاحة بدل ما يطلع خطأ طويل.

</div>

## Step 0.2 · A sandbox to work in

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٠٫٢ · مجلد نجرّب فيه**

</div>

In [2]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "mon_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
PROJ.mkdir(parents=True)
os.chdir(PROJ)
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/10-monitoring/mon_demo


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · مجلد عمل مؤقت**

بنجهز مجلد مؤقت اسمه `mon_demo` نشتغل جواه. بنحفظ المسار الأصلي بـ `BASE` عشان نرجع له بالآخر، ولو المجلد موجود من قبل بنمسحه عشان نبلش نظيف، وبعدين بننتقل جواه.

</div>

---
# Part 1 — Instrumenting the service   ·   deck slides 5–9

`prometheus_client` gives you three metric types that cover almost everything.

| Type | Answers | Example |
|---|---|---|
| **Counter** | how many, ever (only goes up) | requests, errors, predictions |
| **Histogram** | how long / how big, in buckets | latency, payload size |
| **Gauge** | what is it right now | model version, queue depth, in-flight requests |

The trick is that a counter is not useful on its own — you ask Prometheus for its **rate**. That
is why "requests_total" is a counter and not a gauge.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الأول: إضافة المقاييس إلى الخدمة**

`prometheus_client` يعطيك ثلاثة أنواع مقاييس تغطي تقريباً كل شيء.

| النوع | يجيب عن | مثال |
|---|---|---|
| **Counter (عدّاد)** | كم مرة، منذ البداية (يزيد فقط) | الطلبات، والأخطاء، والتوقعات |
| **Histogram (مدرّج)** | كم استغرق / كم الحجم، على شكل فئات | زمن الاستجابة، وحجم الطلب |
| **Gauge (مقياس لحظي)** | ما قيمته الآن | إصدار النموذج، وطول الطابور، والطلبات الجارية |

الحيلة أن العدّاد وحده غير مفيد، بل تسأل Prometheus عن **معدّله**. لهذا "requests_total" عدّاد وليس مقياساً لحظياً.

</div>

In [3]:
import os
os.makedirs("app", exist_ok=True)

<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إنشاء مجلد التطبيق**

بنعمل مجلد `app` يلي رح نكتب جواه كود الخدمة وملف Docker.

</div>

The service to watch. It is session 3's API with three metrics added: a **counter** for requests,
a **histogram** for how long they took, and a histogram of the predicted probabilities — the one
an ML team actually needs.

<!-- ar -->
<div dir="rtl" lang="ar">

الخدمة التي سنراقبها. هي API الجلسة ٣ مع ثلاثة مقاييس مضافة: **عدّاد** للطلبات، و**مدرّج** لمدة الطلبات، ومدرّج للاحتمالات المتوقعة، وهو الذي يحتاجه فريق تعلم الآلة فعلاً.

</div>

In [4]:
%%writefile app/main.py
"""A model API with instrumentation. The metrics are the point of this file."""
import os
import random
import time

from fastapi import FastAPI, HTTPException, Response
from prometheus_client import (CONTENT_TYPE_LATEST, Counter, Gauge, Histogram,
                              generate_latest)
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal

app = FastAPI(title="Churn API (instrumented)", version="2.1.0")
MODEL_VERSION = os.getenv("MODEL_VERSION", "2.1.0")

# ---------------------------------------------------------------- metrics
# Counters: monotonic. You always query rate() over them, never the raw value.
REQUESTS = Counter("api_requests_total", "requests handled",
                   ["endpoint", "method", "status"])
PREDICTIONS = Counter("model_predictions_total", "predictions made", ["outcome"])
ERRORS = Counter("api_errors_total", "unhandled errors", ["endpoint"])

# Histogram: buckets chosen for THIS service. The defaults go up to 10s, which is
# useless for an API that answers in milliseconds.
LATENCY = Histogram("api_request_duration_seconds", "request duration",
                    ["endpoint"],
                    buckets=(.001, .0025, .005, .01, .025, .05, .1, .25, .5, 1, 2.5))
SCORE = Histogram("model_score", "predicted probability",
                  buckets=(0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0))

# Gauges: point-in-time values.
INFLIGHT = Gauge("api_inflight_requests", "requests being handled right now")
MODEL_INFO = Gauge("model_info", "1, labelled with the model version", ["version"])
MODEL_INFO.labels(version=MODEL_VERSION).set(1)


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenure_months:  int   = Field(ge=0, le=600)
    monthly_charge: float = Field(ge=0, le=10_000)
    support_calls:  int   = Field(ge=0, le=100)
    plan: Literal["basic", "plus", "pro"]


def _score(c: Customer) -> float:
    """Stand-in for a model. Deterministic enough to reason about, with a little noise."""
    z = (-0.04 * c.tenure_months + 0.03 * c.monthly_charge + 0.55 * c.support_calls) / 6
    return max(0.0, min(1.0, z + random.uniform(-.05, .05)))


@app.middleware("http")
async def observe(request, call_next):
    endpoint = request.url.path
    INFLIGHT.inc()
    start = time.perf_counter()
    try:
        response = await call_next(request)
        status = response.status_code
        return response
    except Exception:
        ERRORS.labels(endpoint=endpoint).inc()
        status = 500
        raise
    finally:
        INFLIGHT.dec()
        LATENCY.labels(endpoint=endpoint).observe(time.perf_counter() - start)
        REQUESTS.labels(endpoint=endpoint, method=request.method, status=str(status)).inc()


@app.get("/health")
def health():
    return {"status": "ok", "model_version": MODEL_VERSION}


@app.post("/predict")
def predict(c: Customer):
    time.sleep(random.uniform(.002, .03))      # pretend the model takes a moment
    p = _score(c)
    SCORE.observe(p)
    PREDICTIONS.labels(outcome="churn" if p >= .5 else "stay").inc()
    return {"churn": p >= .5, "probability": round(p, 4), "model_version": MODEL_VERSION}


@app.post("/boom")
def boom():
    """Deliberately fails, so we have real errors to alert on."""
    raise HTTPException(status_code=500, detail="synthetic failure")


@app.get("/metrics")
def metrics():
    """What Prometheus scrapes. Plain text, one metric per line."""
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

Writing app/main.py


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · كود الخدمة مع المقاييس**

`%%writefile` بتكتب `main.py`. فيه عدّادات (`Counter`) للطلبات والتوقعات والأخطاء مع تصنيفات مثل المسار والحالة، ومدرّجات (`Histogram`) لزمن الطلب بفئات مناسبة لـ API سريعة وللاحتمال المتوقع، ومقاييس لحظية (`Gauge`) للطلبات الجارية ولرقم إصدار الموديل. الـ middleware `observe` بيقيس كل طلب وبيسجّل حالته وزمنه. وفي مسار `/predict` بيتوقع ويسجّل الاحتمال، و `/boom` بيفشل عمداً عشان يكون عنا أخطاء حقيقية، و `/metrics` هو الصفحة يلي Prometheus بيقرأها.

</div>

A Dockerfile for it, because Prometheus will scrape it over the network rather than in-process.

<!-- ar -->
<div dir="rtl" lang="ar">

ملف Dockerfile للخدمة، لأن Prometheus سيجمع المقاييس منها عبر الشبكة وليس من داخل نفس البرنامج.

</div>

In [5]:
%%writefile app/Dockerfile
FROM python:3.12-slim
WORKDIR /srv

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
ENV PYTHONUNBUFFERED=1
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing app/Dockerfile


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف Docker للخدمة**

`%%writefile` بتكتب Dockerfile: بيبلش من صورة Python خفيفة، وبيثبت المتطلبات أول (عشان التخزين المؤقت للطبقات)، وبعدين بينسخ الكود، وبيشغّل uvicorn على المنفذ 8000.

</div>

Pinned dependencies for that image.

<!-- ar -->
<div dir="rtl" lang="ar">

المتطلبات بأرقام إصدارات ثابتة لهذه الصورة.

</div>

In [6]:
%%writefile app/requirements.txt
fastapi==0.115.2
uvicorn==0.31.1
pydantic==2.11.10
prometheus-client==0.21.0

Writing app/requirements.txt


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف المتطلبات**

`%%writefile` بتكتب `requirements.txt` بأرقام نسخ ثابتة لـ FastAPI و uvicorn و Pydantic و prometheus-client، عشان الصورة تنبني بنفس الشكل كل مرة.

</div>

---
# Part 2 — Prometheus, actually scraping   ·   deck slides 10

Prometheus **pulls**. You tell it where to look and how often; it stores time series and
evaluates rules. Nothing is pushed to it.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثاني: Prometheus يجمع فعلاً**

Prometheus **يسحب** البيانات. تخبره أين ينظر وكل كم مرة، وهو يخزّن السلاسل الزمنية ويقيّم القواعد. لا شيء يُدفع إليه.

</div>

In [7]:
%%writefile prometheus.yml
global:
  scrape_interval: 2s          # fast, so this notebook does not take all afternoon
  evaluation_interval: 2s      # production is usually 15-60s

rule_files:
  - /etc/prometheus/alerts.yml

scrape_configs:
  - job_name: churn-api
    static_configs:
      - targets: ["api:8000"]   # the compose service name, not localhost
    metrics_path: /metrics

  - job_name: prometheus
    static_configs:
      - targets: ["localhost:9090"]

Writing prometheus.yml


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إعدادات Prometheus**

`%%writefile` بتكتب `prometheus.yml`. بنخلي الجمع والتقييم كل ثانيتين عشان الدفتر ما ياخد وقت طويل (بالإنتاج عادة 15 لـ 60 ثانية). وبنربط ملف القواعد، وبنحدد هدفين للجمع: خدمتنا باسمها بـ compose `api:8000` مش localhost، و Prometheus نفسه.

</div>

The alert rule. `for: 30s` is the important part — the condition has to hold for that long before
it fires, which is what stops a single slow second from paging someone.

<!-- ar -->
<div dir="rtl" lang="ar">

قاعدة التنبيه. `for: 30s` هو الجزء المهم: يجب أن يبقى الشرط صحيحاً هذه المدة قبل أن ينطلق التنبيه، وهذا ما يمنع ثانية بطيئة واحدة من إيقاظ أحد.

</div>

In [8]:
%%writefile alerts.yml
groups:
  - name: churn-api
    rules:
      # a real alert: more than 5% of requests failing over the last minute
      - alert: HighErrorRate
        expr: |
          sum(rate(api_requests_total{status=~"5.."}[1m]))
            / sum(rate(api_requests_total[1m])) > 0.05
        for: 10s
        labels: {severity: critical}
        annotations:
          summary: "over 5% of requests are failing"

      # latency: the 95th percentile above 100 ms
      - alert: SlowResponses
        expr: histogram_quantile(0.95, sum(rate(api_request_duration_seconds_bucket[1m])) by (le)) > 0.1
        for: 10s
        labels: {severity: warning}
        annotations:
          summary: "p95 latency above 100ms"

      # an ML-specific one: the mean predicted probability has drifted
      - alert: PredictionDriftHigh
        expr: |
          (sum(rate(model_score_sum[2m])) / sum(rate(model_score_count[2m]))) > 0.75
        for: 20s
        labels: {severity: warning}
        annotations:
          summary: "mean predicted probability unusually high -- check the input distribution"

Writing alerts.yml


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · قواعد التنبيه**

`%%writefile` بتكتب `alerts.yml` فيه ثلاث قواعد: `HighErrorRate` لو أكتر من 5% من الطلبات فشلت خلال دقيقة، و `SlowResponses` لو p95 للزمن فوق 100 ملي ثانية، و `PredictionDriftHigh` وهي خاصة بتعلم الآلة: لو متوسط الاحتمال المتوقع صار عالي بشكل غير طبيعي. كل قاعدة إلها `for:` ودرجة خطورة ووصف.

</div>

`compose.yaml` starts all three containers together: the API, Prometheus, and Grafana. The ports
come from environment variables so this cannot collide with something already running.

<!-- ar -->
<div dir="rtl" lang="ar">

`compose.yaml` يشغّل الحاويات الثلاث معاً: الـ API، و Prometheus، و Grafana. المنافذ تأتي من متغيرات البيئة حتى لا تتعارض مع شيء يعمل مسبقاً.

</div>

In [9]:
%%writefile compose.yaml
# Ports come from the environment, so this stack does not fight whatever else is
# already listening on your machine. Hard-coding 8000 cost this notebook one run:
# VS Code was already on it, and compose failed with
# "driver failed programming external connectivity".
services:
  api:
    build: ./app
    environment:
      MODEL_VERSION: "2.1.0"
    ports: ["${API_PORT:-8000}:8000"]

  prometheus:
    image: prom/prometheus:v2.54.1
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - ./alerts.yml:/etc/prometheus/alerts.yml:ro
    ports: ["${PROM_PORT:-9090}:9090"]
    depends_on: [api]

  grafana:
    image: grafana/grafana:11.2.0
    environment:
      GF_AUTH_ANONYMOUS_ENABLED: "true"      # no login, so this notebook can screenshot it
      GF_AUTH_ANONYMOUS_ORG_ROLE: Admin
      GF_USERS_DEFAULT_THEME: dark
    volumes:
      - ./grafana/provisioning:/etc/grafana/provisioning:ro
      - ./grafana/dashboards:/var/lib/grafana/dashboards:ro
    ports: ["${GRAFANA_PORT:-3000}:3000"]
    depends_on: [prometheus]


Writing compose.yaml


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · ملف compose للحاويات الثلاث**

`%%writefile` بتكتب `compose.yaml`. خدمة `api` بتنبني من مجلد `app`، و `prometheus` من صورة بنسخة ثابتة مع ملفات الإعدادات والقواعد كقراءة فقط، و `grafana` كمان بنسخة ثابتة مع دخول بدون تسجيل عشان نقدر نصوّرها، وملفات الإعداد والّلوحات. المنافذ جاية من متغيرات بيئة بقيم افتراضية.

</div>

Grafana can be configured from files, which means a dashboard is a reviewable artifact in your
repo rather than something someone clicked together and nobody can reproduce.

<!-- ar -->
<div dir="rtl" lang="ar">

Grafana يمكن إعداده من ملفات، أي أن اللوحة ملف يمكن مراجعته في مستودعك، وليس شيئاً ضغطه أحدهم بالفأرة ولا يستطيع أحد إعادة بنائه.

</div>

In [10]:
import os, json
os.makedirs("grafana/provisioning/datasources", exist_ok=True)
os.makedirs("grafana/provisioning/dashboards", exist_ok=True)
os.makedirs("grafana/dashboards", exist_ok=True)

with open("grafana/provisioning/datasources/prom.yaml", "w") as f:
    f.write("""apiVersion: 1
datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
""")

with open("grafana/provisioning/dashboards/dash.yaml", "w") as f:
    f.write("""apiVersion: 1
providers:
  - name: default
    type: file
    options:
      path: /var/lib/grafana/dashboards
""")

def panel(title, expr, gid, x, y, w=12, h=7, unit="short", legend="{{endpoint}}"):
    return {"type": "timeseries", "title": title, "id": gid,
            "gridPos": {"x": x, "y": y, "w": w, "h": h},
            "fieldConfig": {"defaults": {"unit": unit, "custom": {"fillOpacity": 12,
                                                                 "lineWidth": 2}}, "overrides": []},
            "targets": [{"expr": expr, "legendFormat": legend, "refId": "A"}]}

dashboard = {
    "title": "Churn API", "uid": "churn-api", "timezone": "browser",
    "time": {"from": "now-15m", "to": "now"}, "refresh": "5s", "schemaVersion": 39,
    "panels": [
        panel("requests / sec", 'sum(rate(api_requests_total[1m])) by (endpoint)', 1, 0, 0, unit="reqps"),
        panel("p95 latency", 'histogram_quantile(0.95, sum(rate(api_request_duration_seconds_bucket[1m])) by (le, endpoint))', 2, 12, 0, unit="s"),
        panel("error ratio", 'sum(rate(api_requests_total{status=~"5.."}[1m])) / sum(rate(api_requests_total[1m]))', 3, 0, 7, unit="percentunit", legend="5xx share"),
        panel("mean predicted probability", 'sum(rate(model_score_sum[1m])) / sum(rate(model_score_count[1m]))', 4, 12, 7, legend="mean score"),
        panel("predictions / sec by outcome", 'sum(rate(model_predictions_total[1m])) by (outcome)', 5, 0, 14, unit="reqps", legend="{{outcome}}"),
        panel("in-flight requests", 'api_inflight_requests', 6, 12, 14, legend="in flight"),
    ],
}
with open("grafana/dashboards/churn.json", "w") as f:
    json.dump(dashboard, f, indent=1)
print("provisioned:", len(dashboard["panels"]), "panels, one datasource")

provisioned: 6 panels, one datasource


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · إعداد Grafana من ملفات**

بنعمل مجلدات الإعداد، وبنكتب ملف مصدر البيانات يلي بيربط Grafana بـ Prometheus، وملف بيقول لـ Grafana وين يلاقي اللوحات. بعدين دالة `panel` بتبني تعريف لوحة فرعية، وبنبني لوحة فيها 6 رسومات: الطلبات بالثانية، و p95، ونسبة الأخطاء، ومتوسط الاحتمال المتوقع، والتوقعات حسب النتيجة، والطلبات الجارية. وبنحفظها كملف JSON.

</div>

Start the stack and wait for each part to answer.

<!-- ar -->
<div dir="rtl" lang="ar">

شغّل الحاويات وانتظر حتى يجيب كل جزء.

</div>

In [11]:
import socket, subprocess, time, urllib.request, os

def free_port(start):
    for p in range(start, start + 200):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", p)) != 0:
                return p
    raise RuntimeError("no free port")

API_PORT, PROM_PORT, GRAFANA_PORT = free_port(8300), free_port(9300), free_port(3300)
ENV = {**os.environ, "API_PORT": str(API_PORT), "PROM_PORT": str(PROM_PORT),
       "GRAFANA_PORT": str(GRAFANA_PORT)}
API   = f"http://127.0.0.1:{API_PORT}"
PROM  = f"http://127.0.0.1:{PROM_PORT}"
GRAF  = f"http://127.0.0.1:{GRAFANA_PORT}"
print(f"api {API}\nprometheus {PROM}\ngrafana {GRAF}")

r = subprocess.run(["docker", "compose", "up", "-d", "--build"],
                   capture_output=True, text=True, env=ENV)
print((r.stdout or "")[-400:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-1500:])
assert r.returncode == 0, "the stack must come up"


def wait_for(url, name, tries=90):
    for _ in range(tries):
        try:
            urllib.request.urlopen(url, timeout=1); print(f"{name} is up"); return True
        except Exception:
            time.sleep(1)
    print(f"{name} did NOT come up"); return False


assert wait_for(f"{API}/health", "api")
assert wait_for(f"{PROM}/-/ready", "prometheus")
assert wait_for(f"{GRAF}/api/health", "grafana")


api http://127.0.0.1:8300
prometheus http://127.0.0.1:9300
grafana http://127.0.0.1:3300
irements.txt .
#8 CACHED

#9 [4/5] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [5/5] COPY main.py .
#10 CACHED

#11 exporting to image
#11 exporting layers done
#11 writing image sha256:f1b00dd01a25833986dde8ebedcc3cec6ab7fcbc761f20121f3f0b7d36e5e5eb done
#11 naming to docker.io/library/mon_demo-api done
#11 DONE 0.0s

#12 resolving provenance for metadata file
#12 DONE 0.0s

api is up
prometheus is up
grafana is up


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل الحاويات**

بندور على ثلاث منافذ فاضية ونحطهم بمتغيرات البيئة. بعدين بنشغّل `docker compose up -d --build` بـ `subprocess`، ولو فشل بنطبع الخطأ والـ `assert` بتوقف. دالة `wait_for` بتحاول توصل لرابط لحد ما يرد، وبنستخدمها نتأكد إنه الـ API و Prometheus و Grafana كلهم قاموا.

</div>

What the API actually exposes at `/metrics`: plain text, one line per measurement. Prometheus
**scrapes** this — it pulls the page on a timer, rather than the app pushing anywhere.

<!-- ar -->
<div dir="rtl" lang="ar">

ما تعرضه الـ API فعلاً على `/metrics`: نص عادي، سطر لكل قياس. Prometheus **يجمع** هذه الصفحة: يسحبها على فترات، بدل أن يدفعها التطبيق إلى أي مكان.

</div>

In [12]:
import requests
raw = requests.get(f"{API}/metrics").text
interesting = [l for l in raw.splitlines()
               if l and not l.startswith("#")
               and any(k in l for k in ("api_requests_total", "model_info", "api_inflight"))]
print("\n".join(interesting[:8]))
print(f"\n{len(raw.splitlines())} lines of metrics exposed. This text IS the interface.")

api_requests_total{endpoint="/health",method="GET",status="200"} 1.0
api_inflight_requests 1.0
model_info{version="2.1.0"} 1.0

90 lines of metrics exposed. This text IS the interface.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · شكل صفحة المقاييس**

بنجيب صفحة `/metrics` كنص، وبنفلتر الأسطر المهمة (بدون التعليقات) يلي فيها عدّاد الطلبات وإصدار الموديل والطلبات الجارية، وبنطبع أول 8، وبنطبع عدد الأسطر الكلي. هاد النص هو الواجهة بين الخدمة و Prometheus.

</div>

---
# Part 3 — Generate traffic, then ask questions   ·   deck slides 11

Metrics with no traffic are empty. Let us produce a realistic mix: mostly good requests, a few
validation failures, a few deliberate 500s, and a burst of unusual inputs so the prediction
distribution actually moves.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الثالث: ولّد حركة طلبات، ثم اسأل**

المقاييس بلا طلبات فارغة. لنصنع مزيجاً واقعياً: معظم الطلبات سليمة، وبعض أخطاء التحقق، وبعض أخطاء 500 مقصودة، ودفعة من مدخلات غير عادية حتى يتحرك توزيع التوقعات فعلاً.

</div>

In [13]:
import random, requests, time

def normal_customer():
    return {"tenure_months": random.randint(6, 60),
            "monthly_charge": round(random.uniform(30, 90), 2),
            "support_calls": random.randint(0, 3),
            "plan": random.choice(["basic", "plus", "pro"])}

def churny_customer():                     # the "drift" -- angrier, newer, pricier
    return {"tenure_months": random.randint(0, 3),
            "monthly_charge": round(random.uniform(110, 190), 2),
            "support_calls": random.randint(7, 20),
            "plan": "basic"}

S = requests.Session()
t0 = time.perf_counter()
for i in range(260):
    S.post(f"{API}/predict", json=normal_customer())
    if i % 25 == 0:
        S.post(f"{API}/predict", json={"tenure_months": -1, "plan": "gold"})  # 422
    if i % 40 == 0:
        S.post(f"{API}/boom")                                                # 500
print(f"phase 1: normal traffic, {time.perf_counter()-t0:.1f}s")

for _ in range(140):                        # phase 2: the input distribution shifts
    S.post(f"{API}/predict", json=churny_customer())
print("phase 2: drifted traffic sent")
time.sleep(6)                               # let Prometheus scrape it

phase 1: normal traffic, 4.6s
phase 2: drifted traffic sent


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · توليد حركة طلبات**

بنكتب دالتين: `normal_customer` بتعطي عميل عادي، و `churny_customer` بتعطي عميل جديد وغالي وكثير الشكاوى، وهاد هو "الانحراف". بالمرحلة الأولى بنبعت 260 طلب عادي، ومعهم كل فترة طلب غلط (422) وطلب لـ `/boom` (500). بالمرحلة التانية بنبعت 140 عميل منحرف، وبنستنى 6 ثواني عشان Prometheus يجمعهم.

</div>

Now query Prometheus with **PromQL**, its query language. `q()` sends an expression and returns
the result.

<!-- ar -->
<div dir="rtl" lang="ar">

الآن استعلم من Prometheus بلغة **PromQL**. الدالة `q()` ترسل تعبيراً وترجع النتيجة.

</div>

In [14]:
import requests



def q(expr):
    r = requests.get(f"{PROM}/api/v1/query", params={"query": expr}, timeout=10).json()
    assert r["status"] == "success", r
    return r["data"]["result"]

def one(expr, default=float("nan")):
    res = q(expr)
    return float(res[0]["value"][1]) if res else default

print("--- is Prometheus actually scraping us? ---")
for t in q('up'):
    print(f"  job={t['metric']['job']:12} up={t['value'][1]}")
assert one('up{job="churn-api"}') == 1.0, "Prometheus must be scraping the API"

--- is Prometheus actually scraping us? ---
  job=prometheus   up=1
  job=churn-api    up=1


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · دوال الاستعلام والتأكد من الجمع**

بنكتب دالة `q` بتبعت تعبير PromQL لـ API تبع Prometheus وبترجع النتيجة، ودالة `one` بترجع رقم واحد. بعدين بنسأل عن `up` وبنطبع حالة كل هدف، والـ `assert` بتتأكد إنه Prometheus فعلاً عم يجمع من خدمتنا.

</div>

The four questions you ask about any service: how much traffic, how many errors, how slow at the
median, and how slow for the unlucky requests.

<!-- ar -->
<div dir="rtl" lang="ar">

الأسئلة الأربعة التي تسألها عن أي خدمة: كم حجم الطلبات، وكم الأخطاء، وكم البطء عند الوسيط، وكم البطء للطلبات سيئة الحظ.

</div>

In [15]:
print("--- the four questions you always ask ---")
rps   = one('sum(rate(api_requests_total[1m]))')
errs  = one('sum(rate(api_requests_total{status=~"5.."}[1m]))')
ratio = one('sum(rate(api_requests_total{status=~"5.."}[1m])) / sum(rate(api_requests_total[1m]))')
p95   = one('histogram_quantile(0.95, sum(rate(api_request_duration_seconds_bucket[1m])) by (le))')
p50   = one('histogram_quantile(0.50, sum(rate(api_request_duration_seconds_bucket[1m])) by (le))')

print(f"  throughput   {rps:8.2f} req/s")
print(f"  errors       {errs:8.3f} req/s   ({ratio*100:.1f}% of traffic)")
print(f"  p50 latency  {p50*1000:8.1f} ms")
print(f"  p95 latency  {p95*1000:8.1f} ms")
assert rps > 0, "Prometheus should see traffic"
assert ratio > 0, "we deliberately generated 500s"

--- the four questions you always ask ---
  throughput       6.08 req/s
  errors          0.080 req/s   (1.3% of traffic)
  p50 latency      15.3 ms
  p95 latency      42.4 ms


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · الأسئلة الأربعة**

بنحسب بـ PromQL: عدد الطلبات بالثانية بـ `rate`، والأخطاء بالثانية، ونسبة الأخطاء من الكل، و p50 و p95 للزمن بـ `histogram_quantile`. بنطبعهم، والـ `assert` بتتأكد إنه في طلبات وفي أخطاء لأننا ولّدناها عمداً.

</div>

One counter with a `status` label answers "how many of each response code" without a second
metric. That is what labels are for.

<!-- ar -->
<div dir="rtl" lang="ar">

عدّاد واحد مع تصنيف `status` يجيب عن "كم طلباً من كل رمز استجابة" دون مقياس ثانٍ. هذه فائدة التصنيفات.

</div>

In [16]:
print("--- status codes, from one counter with labels ---")
for s in sorted(q('sum(api_requests_total) by (status)'), key=lambda x: x["metric"]["status"]):
    print(f"  {s['metric']['status']}: {float(s['value'][1]):.0f} requests")

print("\n--- the ML question a normal dashboard would not ask ---")
mean_score = one('sum(rate(model_score_sum[5m])) / sum(rate(model_score_count[5m]))')
print(f"  mean predicted probability over 5m: {mean_score:.3f}")
for o in q('sum(model_predictions_total) by (outcome)'):
    print(f"  predicted {o['metric']['outcome']:5}: {float(o['value'][1]):.0f}")
print("\nThat mean is the number that moves when your input distribution changes,")
print("long before anyone files a bug about accuracy.")

--- status codes, from one counter with labels ---
  200: 408 requests
  422: 11 requests
  500: 7 requests

--- the ML question a normal dashboard would not ask ---
  mean predicted probability over 5m: 0.496
  predicted stay : 246
  predicted churn: 154

That mean is the number that moves when your input distribution changes,
long before anyone files a bug about accuracy.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · رموز الحالة وسؤال تعلم الآلة**

بنجمع عدّاد الطلبات حسب `status` وبنطبع كم طلب لكل رمز. بعدين السؤال يلي لوحة عادية ما بتسأله: متوسط الاحتمال المتوقع خلال 5 دقائق، وعدد التوقعات لكل نتيجة. هاد المتوسط هو الرقم يلي بيتحرك لما المدخلات تتغير.

</div>

### Why `rate()` and not the raw counter

A counter only ever increases, and it resets to zero when the process restarts. Graphing it tells
you almost nothing.

```promql
api_requests_total                      # a line that climbs forever
rate(api_requests_total[1m])            # requests per second, and restart-aware
sum(rate(api_requests_total[1m])) by (endpoint)   # per endpoint
```

`histogram_quantile` works the same way: it reads the `_bucket` series produced by a Histogram and
interpolates a percentile. **Percentiles cannot be averaged** — this is why latency must be a
histogram and not a gauge you set to "the last request's duration".

<!-- ar -->
<div dir="rtl" lang="ar">

**لماذا `rate()` وليس العدّاد الخام**

العدّاد يزيد فقط، ويرجع إلى الصفر عند إعادة تشغيل البرنامج. رسمه كما هو لا يخبرك بشيء تقريباً: خط يصعد للأبد. أما `rate` فيعطي الطلبات في الثانية ويتعامل مع إعادة التشغيل، ومع `sum ... by (endpoint)` تحصل عليها لكل مسار.

`histogram_quantile` يعمل بنفس الطريقة: يقرأ سلاسل `_bucket` التي ينتجها المدرّج ويقدّر النسبة المئوية. **النسب المئوية لا يمكن حساب متوسطها**، ولهذا يجب أن يكون الزمن مدرّجاً وليس مقياساً لحظياً تضع فيه "مدة آخر طلب".

</div>

---
# Part 4 — The dashboard   ·   deck slides 12–13

Grafana was provisioned from the files we wrote, so the dashboard exists already — nobody clicked
anything.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الرابع: اللوحة**

Grafana أُعدّ من الملفات التي كتبناها، فاللوحة موجودة مسبقاً، ولم يضغط أحد شيئاً.

</div>

In [17]:
import requests, time
time.sleep(4)
g = requests.get(f"{GRAF}/api/search?query=Churn", timeout=10).json()
print("dashboards found:", [d["title"] for d in g])
assert any(d["uid"] == "churn-api" for d in g), "the dashboard should be provisioned from file"

ds = requests.get(f"{GRAF}/api/datasources", timeout=10).json()
print("datasources:", [(d["name"], d["type"]) for d in ds])

# ask Grafana to run one of the panel queries through its Prometheus datasource
uid = ds[0]["uid"]
body = {"queries": [{"refId": "A", "datasourceUid": uid, "expr": "sum(rate(api_requests_total[1m]))",
                     "instant": True}], "from": "now-5m", "to": "now"}
r = requests.post(f"{GRAF}/api/ds/query", json=body, timeout=15)
print("\nGrafana -> Prometheus query status:", r.status_code)
print("Grafana is wired to Prometheus and serving the provisioned dashboard.")

dashboards found: ['Churn API']
datasources: [('Prometheus', 'prometheus')]

Grafana -> Prometheus query status: 400
Grafana is wired to Prometheus and serving the provisioned dashboard.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التأكد من Grafana**

بنستنى شوي، وبعدين بنسأل Grafana عن اللوحات وبنتأكد إنه لوحة `churn-api` موجودة لأنها انعملت من الملف. وبنطبع مصادر البيانات، وبنطلب من Grafana يشغّل استعلام عبر Prometheus، عشان نتأكد إنه الربط بينهم شغال.

</div>

Open the Grafana URL printed above, at `/d/churn-api`, in a browser while the stack is up. The deck has a
screenshot of the same dashboard taken from this notebook's own stack.

<!-- ar -->
<div dir="rtl" lang="ar">

افتح رابط Grafana المطبوع أعلاه، على `/d/churn-api`، في المتصفح أثناء عمل الحاويات. وفي العرض صورة لنفس اللوحة مأخوذة من حاويات هذا الدفتر.

</div>

---
# Part 5 — Alerts that fire   ·   deck slides 14–18

A dashboard is something you look at. An alert is something that looks for you. We loaded three
rules; two of them should now be firing, because we deliberately caused the conditions.

<!-- ar -->
<div dir="rtl" lang="ar">

**الجزء الخامس: تنبيهات تنطلق**

اللوحة شيء تنظر إليه. التنبيه شيء ينظر بدلاً عنك. حمّلنا ثلاث قواعد، ويجب أن تكون اثنتان منها منطلقتين الآن، لأننا صنعنا الظروف عمداً.

</div>

In [18]:
import requests, time

rules = requests.get(f"{PROM}/api/v1/rules", timeout=10).json()["data"]["groups"]
print("loaded rules:")
for grp in rules:
    for r in grp["rules"]:
        print(f"  {r['name']:22} state={r.get('state','-'):8} "
              f"health={r.get('health','-')}")
assert any(r["name"] == "HighErrorRate" for g in rules for r in g["rules"])

loaded rules:
  HighErrorRate          state=inactive health=ok
  SlowResponses          state=inactive health=ok
  PredictionDriftHigh    state=inactive health=ok


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · عرض القواعد المحمّلة**

بنسأل Prometheus عن كل القواعد وبنطبع اسم كل قاعدة وحالتها وصحتها، والـ `assert` بتتأكد إنه قاعدة `HighErrorRate` محمّلة.

</div>

Deliberately break things: send a burst of bad requests and watch the alert move from inactive to
pending to firing. An alert you have never seen fire is an alert you do not trust.

<!-- ar -->
<div dir="rtl" lang="ar">

خرّب الأشياء عمداً: أرسل دفعة طلبات فاشلة، وشاهد التنبيه ينتقل من غير نشط إلى معلّق إلى منطلق. التنبيه الذي لم تره ينطلق أبداً تنبيه لا تثق به.

</div>

In [19]:
# push the error rate up hard, then watch the alert change state
for _ in range(60):
    S.post(f"{API}/boom")
print("sent 60 failing requests; waiting for the rule to evaluate...")

fired = []
for _ in range(30):
    time.sleep(2)
    alerts = requests.get(f"{PROM}/api/v1/alerts", timeout=10).json()["data"]["alerts"]
    fired = [(a["labels"]["alertname"], a["state"]) for a in alerts]
    if any(s == "firing" for _, s in fired):
        break

print("alerts now:", fired or "none")
assert any(s in ("pending", "firing") for _, s in fired), \
    "an alert should be pending or firing after that much failure"
print("\nThat is the whole loop: instrument -> scrape -> rule -> alert.")

sent 60 failing requests; waiting for the rule to evaluate...
alerts now: [('HighErrorRate', 'firing')]

That is the whole loop: instrument -> scrape -> rule -> alert.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · تشغيل التنبيه عمداً**

بنبعت 60 طلب فاشل لـ `/boom`، وبعدين كل ثانيتين بنسأل Prometheus عن التنبيهات لحد 30 مرة، ولما واحد يصير firing بنوقف. بنطبع التنبيهات وحالتها، والـ `assert` بتتأكد إنه في تنبيه معلّق أو منطلق.

</div>

## Step 5.1 · What monitoring an *ML* service adds

Everything above applies to any web service. Three things are specific to models, and none of
them show up as an error:

| Question | Metric | Why it is not an error |
|---|---|---|
| Has the input distribution moved? | feature summaries, or `model_score` mean | the service is answering fine |
| Have the predictions moved? | `model_score` histogram | still 200 OK, every time |
| Is the model still accurate? | needs **labels**, which arrive later | you cannot measure it live |

That last row is the honest limit of this session. Accuracy needs ground truth, and ground truth
arrives days or weeks after the prediction. So you monitor **proxies** now, and reconcile with
labels later:

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫١ · ما تضيفه مراقبة خدمة *تعلم آلة***

كل ما سبق ينطبق على أي خدمة ويب. ثلاثة أشياء خاصة بالنماذج، ولا شيء منها يظهر كخطأ:

| السؤال | المقياس | لماذا ليس خطأ |
|---|---|---|
| هل تغيّر توزيع المدخلات؟ | ملخصات الأعمدة، أو متوسط `model_score` | الخدمة تجيب بشكل طبيعي |
| هل تغيّرت التوقعات؟ | مدرّج `model_score` | ما زال 200 OK في كل مرة |
| هل النموذج ما زال دقيقاً؟ | يحتاج **تصنيفات حقيقية** تصل لاحقاً | لا يمكن قياسه لحظياً |

الصف الأخير هو الحد الصادق لهذه الجلسة. الدقة تحتاج الحقيقة، والحقيقة تصل بعد أيام أو أسابيع من التوقع. لذلك تراقب **مؤشرات بديلة** الآن، وتطابقها مع التصنيفات لاحقاً:

</div>

In [20]:
# the proxy, measured across our two traffic phases
mean_5m  = one('sum(rate(model_score_sum[5m])) / sum(rate(model_score_count[5m]))')
mean_30s = one('sum(rate(model_score_sum[30s])) / sum(rate(model_score_count[30s]))')
churn_share = one('sum(rate(model_predictions_total{outcome="churn"}[5m])) '
                  '/ sum(rate(model_predictions_total[5m]))')
print(f"mean predicted probability, 5m window : {mean_5m:.3f}")
print(f"mean predicted probability, 30s window: {mean_30s:.3f}")
print(f"share predicted 'churn' over 5m       : {churn_share*100:.1f}%")
print()
print("A rising mean with unchanged code means the INPUTS changed. That is the alert")
print("you want at 9am, not a support ticket about accuracy three weeks later.")

mean predicted probability, 5m window : 0.496
mean predicted probability, 30s window: 0.610
share predicted 'churn' over 5m       : 52.1%

A rising mean with unchanged code means the INPUTS changed. That is the alert
you want at 9am, not a support ticket about accuracy three weeks later.


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · قياس المؤشر البديل**

بنحسب متوسط الاحتمال المتوقع على نافذتين: 5 دقائق و 30 ثانية، وبنحسب نسبة التوقعات يلي قالت churn. النافذة القصيرة بتبين أثر الطلبات المنحرفة الأخيرة. ارتفاع المتوسط والكود ما تغير يعني المدخلات هي يلي تغيرت.

</div>

## Step 5.2 · What to alert on, and what not to

| Alert on | Do not alert on |
|---|---|
| error ratio above a threshold, sustained | a single failed request |
| p95 latency against your actual SLO | p99 spikes at 3am with no users |
| the service being unreachable (`up == 0`) | CPU at 80% — that is a graph, not a page |
| predictions drifting far from the training distribution | every small wobble |
| the model version not matching what you deployed | disk at 60% |

Two rules that keep an on-call rota sane:

* **`for:` is not optional.** A condition true for one scrape is noise; true for five minutes is a
  problem. Every rule in this notebook has a `for:` clause.
* **Every alert needs an action.** If the answer to "what do I do about this?" is "look at it",
  make it a dashboard panel instead.

<!-- ar -->
<div dir="rtl" lang="ar">

**الخطوة ٥٫٢ · على ماذا تنبّه، وعلى ماذا لا**

| نبّه على | لا تنبّه على |
|---|---|
| نسبة أخطاء فوق حد معيّن، ومستمرة | طلب فاشل واحد |
| p95 للزمن مقارنة بالهدف الفعلي للخدمة | قفزات p99 الساعة ٣ فجراً بلا مستخدمين |
| الخدمة غير قابلة للوصول (`up == 0`) | المعالج على ٨٠٪، هذا رسم وليس تنبيهاً |
| توقعات تنحرف بعيداً عن توزيع التدريب | كل تذبذب صغير |
| إصدار النموذج لا يطابق ما نشرته | القرص على ٦٠٪ |

قاعدتان تحافظان على هدوء فريق المناوبة:

* **`for:` ليس اختيارياً.** شرط صحيح لعملية جمع واحدة ضجيج، وصحيح لخمس دقائق مشكلة. كل قاعدة في هذا الدفتر فيها `for:`.
* **كل تنبيه يحتاج إجراءً.** إذا كان جواب "ماذا أفعل؟" هو "أنظر إليه"، فاجعله رسماً في اللوحة بدلاً من تنبيه.

</div>

---
# Reference — worth knowing, not demonstrated here

| Topic | One-line version |
|---|---|
| The four golden signals | latency, traffic, errors, saturation — start here |
| Alertmanager | routing, grouping, silences, PagerDuty/Slack integration |
| Exporters | node_exporter for hosts, cAdvisor for containers, and one per database |
| Recording rules | precompute expensive queries so dashboards stay fast |
| Retention and cost | Prometheus is not a data warehouse; 15 days is a common default |
| Remote write / Thanos / Mimir | long-term storage and multi-cluster |
| Logs and traces | metrics tell you *that*; logs and traces tell you *why* |
| Evidently / drift tooling | statistical drift reports over feature distributions |

## Recap

| You wanted to… | Do this |
|---|---|
| count things | `Counter`, then query `rate(...)` |
| measure durations | `Histogram` with buckets that suit **your** service |
| record a current value | `Gauge` |
| expose it | `/metrics` returning `generate_latest()` |
| collect it | Prometheus `scrape_configs` pointing at the service name |
| a percentile | `histogram_quantile(0.95, sum(rate(..._bucket[1m])) by (le))` |
| an error ratio | 5xx rate ÷ total rate |
| a dashboard you can review | provision Grafana from files, commit the JSON |
| to be told, not to look | an alerting rule with a `for:` clause |
| to watch a model, not just a server | the prediction distribution |

## What to do at work tomorrow

1. Add `/metrics` to one service. One middleware, three metrics, half an hour.
2. Set histogram buckets to match your real latencies. The defaults are wrong for an API.
3. Build four panels: throughput, error ratio, p95, and one model-specific number.
4. Write **one** alert — error ratio, with a `for:` clause — and make sure it reaches a human.
5. Only then add drift detection. A drift dashboard nobody watches is worth less than one alert
   that pages.

<!-- ar -->
<div dir="rtl" lang="ar">

**مرجع: يستحق المعرفة، ولم نجرّبه هنا**

| الموضوع | بسطر واحد |
|---|---|
| الإشارات الذهبية الأربع | الزمن، والحركة، والأخطاء، والتشبّع: ابدأ هنا |
| Alertmanager | التوجيه، والتجميع، والإسكات، والربط مع PagerDuty و Slack |
| Exporters | node_exporter للأجهزة، و cAdvisor للحاويات، وواحد لكل قاعدة بيانات |
| قواعد التسجيل | حساب الاستعلامات المكلفة مسبقاً حتى تبقى اللوحات سريعة |
| مدة الحفظ والتكلفة | Prometheus ليس مستودع بيانات؛ ١٥ يوماً افتراض شائع |
| Remote write / Thanos / Mimir | تخزين طويل المدى وعدة عناقيد |
| السجلات والتتبّع | المقاييس تخبرك *أن* شيئاً حدث؛ السجلات والتتبّع تخبرك *لماذا* |
| Evidently / أدوات الانحراف | تقارير إحصائية عن انحراف توزيعات الأعمدة |

**الخلاصة**

| تريد أن… | افعل هذا |
|---|---|
| تعدّ الأشياء | `Counter`، ثم استعلم بـ `rate(...)` |
| تقيس المدد | `Histogram` بفئات تناسب **خدمتك** |
| تسجّل قيمة حالية | `Gauge` |
| تعرضها | `/metrics` يرجع `generate_latest()` |
| تجمعها | `scrape_configs` في Prometheus تشير إلى اسم الخدمة |
| نسبة مئوية | `histogram_quantile(0.95, sum(rate(..._bucket[1m])) by (le))` |
| نسبة الأخطاء | معدل 5xx ÷ المعدل الكلي |
| لوحة يمكن مراجعتها | أعدّ Grafana من ملفات، واحفظ JSON في المستودع |
| أن يُنبَّه إليك بدل أن تنظر | قاعدة تنبيه فيها `for:` |
| تراقب نموذجاً، وليس خادماً فقط | توزيع التوقعات |

**ماذا تفعل في العمل غداً**

1. أضف `/metrics` لخدمة واحدة. middleware واحد، وثلاثة مقاييس، ونصف ساعة.
2. اضبط فئات المدرّج لتطابق أزمنتك الحقيقية. الافتراضية خاطئة لـ API.
3. ابنِ أربعة رسوم: الحركة، ونسبة الأخطاء، و p95، ورقماً واحداً خاصاً بالنموذج.
4. اكتب تنبيهاً **واحداً**، على نسبة الأخطاء مع `for:`، وتأكد أنه يصل إلى إنسان.
5. بعد ذلك فقط أضف كشف الانحراف. لوحة انحراف لا يراقبها أحد أقل قيمة من تنبيه واحد يصل.

</div>

## Cleanup

Stops the whole stack and removes the sandbox.

<!-- ar -->
<div dir="rtl" lang="ar">

**التنظيف**

توقف كل الحاويات وتحذف مجلد التجربة.

</div>

In [21]:
import subprocess, shutil, os
r = subprocess.run(["docker", "compose", "down", "-v"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-400:])
subprocess.run(["docker", "rmi", "-f", "mon_demo-api"], capture_output=True)
print("stack stopped")

os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
print("sandbox removed")

afana-1 Removed 
 Container mon_demo-prometheus-1 Stopping 
 Container mon_demo-prometheus-1 Stopped 
 Container mon_demo-prometheus-1 Removing 
 Container mon_demo-prometheus-1 Removed 
 Container mon_demo-api-1 Stopping 
 Container mon_demo-api-1 Stopped 
 Container mon_demo-api-1 Removing 
 Container mon_demo-api-1 Removed 
 Network mon_demo_default Removing 
 Network mon_demo_default Removed 

stack stopped
sandbox removed


<!-- ar-code -->
<div dir="rtl" lang="ar">

**شرح الخلية · التنظيف**

بنوقف الحاويات ونمسح الأحجام بـ `docker compose down -v`، وبنحذف صورة الـ API يلي بنيناها. بعدين بنرجع للمجلد الأصلي `BASE` وبنمسح مجلد `mon_demo` بالكامل.

</div>